In [ ]:
import tensorflow as tf
import numpy as np
import imghdr
import cv2
import os
import matplotlib.pyplot as plt
from tensorflow.keras.layers import Conv2D,MaxPool2D,Dense,Flatten,InputLayer,BatchNormalization
from tensorflow.keras.metrics import BinaryCrossentropy,categorical_crossentropy,sparse_categorical_crossentropy,Recall,Precision
from tensorflow.keras.optimizers import Adam

/tmp/ipython-input-983148801.py:3: DeprecationWarning: 'imghdr' is deprecated and slated for removal in Python 3.13
  import imghdr


In [ ]:
# data_dir="/content/drive/MyDrive/Projects /skin dataset"

In [ ]:
# os.listdir(os.path.join(data_dir,'train'))

In [ ]:
#image_ext=['jpeg','png','jpg','bmp']
# for image_class in os.listdir(data_dir):
#   for image in os.listdir(os.path.join(data_dir,image_class)):
#     image_path=os.path.join(data_dir,image_class,image)
#     try:
#       img = cv2.imread(image_path)
#       tip = imghdr.what(image_path)
#       if tip not in image_ext:
#         print('image not in ext list {}'.format(image_path))
#         os.remove(image_path)
#     except Exception as e:
#       print("issue with image {}".format{image_Path})

In [ ]:
# img=cv2.imread(os.path.join('data','train','Melanoma','/content/drive/MyDrive/Projects /skin dataset/train/Melanoma/ISIC_0000004.jpg'))

In [ ]:
# img.shape

In [ ]:
# plt.imshow(img)

##Data Loading

In [ ]:
data =tf.keras.utils.image_dataset_from_directory("/content/drive/MyDrive/Projects /skin_dataset")

NotFoundError: Could not find directory /content/drive/MyDrive/Projects /skin_dataset

In [ ]:
data_iterator=data.as_numpy_iterator()

In [ ]:
batch = data_iterator.next()

In [ ]:
print(type(batch))
print(len(batch))
for idx, element in enumerate(batch):
    print(f"Element {idx}: {element.shape if hasattr(element,'s hape') else element}")


In [ ]:
batch[0].shape

In [ ]:

print(batch[0])

In [ ]:
fig,ax = plt.subplots(ncols=4,figsize=(20,20))
for idx,img in enumerate(batch[0][:4]):
  ax[idx].imshow(img.astype(int))
  ax[idx].title.set_text(batch[1][idx])

##pre processing

In [ ]:
#scaling data
data = data.map(lambda x,y: (x/255,y))
# val_data=v_data.map(lambda x,y: (x/255,y))

In [ ]:

scaled_iterator = data.as_numpy_iterator()
# vscal_iterate=val_data.as_numpy_iterator()

In [ ]:
batch = scaled_iterator.next()
# val_batch=vscal_iterate.next()

In [ ]:
batch[0].shape
# val_batch[0].shape

In [ ]:
fig, ax = plt.subplots(ncols=4,figsize=(20,20))
for idx, img in enumerate(batch[0][:4]):
  ax[idx].imshow(img)
  ax[idx].title.set_text(batch[1][idx])

In [ ]:
# fig, ax = plt.subplots(ncols=4,figsize=(20,20))
# for idx, img in enumerate(val_batch[0][:4]):
#   ax[idx].imshow(img)
#   ax[idx].title.set_text(val_batch[1][idx])

In [ ]:
len(data)

##spliting data in train validation and testing

In [ ]:
def splits(dataset,TRAIN_RATIO,VAL_RATIO,TEST_RATIO):
  DATASET_SIZE = len(dataset)

  train_dataset= dataset.take(int(TRAIN_RATIO*DATASET_SIZE))

  val_test_dataset= dataset.skip(int(TRAIN_RATIO*DATASET_SIZE))
  val_dataset=val_test_dataset.take(int(VAL_RATIO*DATASET_SIZE))

  test_dataset= val_test_dataset.skip(int(VAL_RATIO*DATASET_SIZE))
  return train_dataset,val_dataset,test_dataset

In [ ]:
# train_size = int(len(data)*.7)
# test_size = int(len(data)*.1)+1
# val_size = int(len(val_data))

TRAIN_RATIO=0.9
VAL_RATIO=0.1
TEST_RATIO=0.1

# dataset = tf.data.Dataset.range(10)
train_dataset,val_dataset,test_dataset = splits(data,TRAIN_RATIO,VAL_RATIO,TEST_RATIO)
# print(list(train_dataset.take(1).as_numpy_iterator()),list(val_dataset.take(1).as_numpy_iterator()),list(test_dataset.take(1).as_numpy_iterator()))

In [ ]:
len(test_dataset)

In [ ]:
len(val_dataset)

In [ ]:
len(train_dataset)

In [ ]:

train_dataset = train_dataset.shuffle(buffer_size=8,reshuffle_each_iteration=True)
val_dataset = val_dataset.shuffle(buffer_size=8,reshuffle_each_iteration=True)

In [ ]:
# train = data.take(train_size)
# val = val_data.take(val_size)
# test = data.skip(train_size).take(test_size)

##building deep learning model

In [ ]:
IM_SIZE=256
model = tf.keras.Sequential([
    InputLayer(shape=(IM_SIZE,IM_SIZE,3)),

    Conv2D(filters= 6, kernel_size=3 ,strides=1, padding='valid',activation='relu'),
    BatchNormalization(),
    MaxPool2D(pool_size= 2,strides=2),

    Conv2D(filters=16,kernel_size=3, strides=1,padding='valid',activation='relu'),
    BatchNormalization(),
    MaxPool2D(pool_size=2,strides=2),

    Flatten(),
    Dense(100,activation='relu'),
    BatchNormalization(),
    Dense(32,activation='relu'),
    BatchNormalization(),
    Dense(8,activation='sigmoid')
])

model.summary()

In [ ]:
model.compile('adam',  loss='sparse_categorical_crossentropy',metrics=['accuracy'])

##Training

In [ ]:
# logdir='/content/drive/MyDrive/Projects /logs'

In [ ]:
# tensorboard_callback= tf.keras.callbacks.TensorBoard(log_dir=logdir)

In [ ]:
 hist = model.fit(train_dataset,validation_data=val_dataset,epochs=20,verbose=1)

In [ ]:
fig = plt.figure()
plt.plot(hist.history['loss'],color='teal',label='loss')
plt.plot(hist.history['val_loss'],color='orange',label='val_loss')
fig.suptitle('Loss',fontsize=20)
plt.legend(loc='upper left')
plt.show()

In [ ]:
fig = plt.figure()
plt.plot(hist.history['accuracy'],color='teal',label='accuracy')
plt.plot(hist.history['val_accuracy'],color='orange',label='val_accuracy')
fig.suptitle('Accuracy',fontsize=20)
plt.legend(loc='upper left')
plt.show()

In [ ]:
test_dataset

In [ ]:
model.evaluate(test_dataset)

In [ ]:
model.predict(test_dataset.take(1))[0][1]

In [ ]:
model.save('/content/drive/MyDrive/Projects /skin_model.h5')